# **Evaluation Notebook**

### **Wikiart Project**

**Group x:**\
**Afonso Hermenegildo** - 202 | **Lara Santos** - 20221823 | **Marco Martins** - 20221951 | **André Nicolau** - 2022

Github repository: https://github.com/MarcoAFMartins/Wikiart_Project

---

# Table of Contents

1. [Classification report (accuracy, macro-F1, per-class precision/recall)](#section-1)  
2. [Confusion matrix (seaborn heatmap)](#section-2)  
3. [Training curves (loss & accuracy)](#section-3)  
4. [Grad-CAM visualisation](#section-4)  
5. [Misclassified samples analysis](#section-5)
6. [Model comparison table](#section-5)

---

# Imports

In [2]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import cv2
import keras

# Add the project to the path so it can find 'notebooks'
sys.path.append('/content/Wikiart_Project')

import keras
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from keras.utils import image_dataset_from_directory
from notebooks.baseline_model import BaselineModel

# Tell Python to look inside your project folder for modules
project_path = "/content/Wikiart_Project"

if project_path not in sys.path:
    sys.path.append(project_path)

# Paths — adjust if needed
DATA_DIR    = Path('Data')
OUTPUTS_DIR = Path('outputs')
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODULES_DIR  = OUTPUTS_DIR / 'modules'
NOTEBOOK_DIR = Path('notebooks')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
SEED        = 42

ModuleNotFoundError: No module named 'notebooks'

# Load dataset

In [ ]:
train_df = image_dataset_from_directory(
    DATA_DIR / "Train",
    label_mode="categorical",
    image_size=(512, 512),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True,
)

val_df = image_dataset_from_directory(
    DATA_DIR / "Validation",
    label_mode="categorical",
    image_size=(512, 512),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False,
)

test_df = image_dataset_from_directory(
    DATA_DIR / "Test",
    label_mode="categorical",
    image_size=(512, 512),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False,
)

class_names = train_df.class_names
print(f"Classes: {len(class_names)} — {class_names}")

Found 9329 files belonging to 23 classes.
Found 1999 files belonging to 23 classes.
Found 2000 files belonging to 23 classes.
Classes: 23 — ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


Because we've done our one subclassed model, model.build() often fails to statically infer shapes through complex internal layers like data augmentation. 
By running a dummy image through the network instead, we force a real forward pass that guarantees every layer dynamically initializes its weights based on the actual tensor size.

In [ ]:
keras.backend.clear_session()
keras.mixed_precision.set_global_policy('mixed_float16')

@keras.saving.register_keras_serializable()
class ConvNeXtLarge3Pretrained(keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.base_model = keras.applications.ConvNeXtLarge(
            include_top=False,
            weights=None,
            input_shape=(512, 512, 3),
        )
        self.base_model.trainable = False
        self.global_max_pooling = keras.layers.GlobalMaxPooling2D()
        self.dropout_layer = keras.layers.Dropout(0.2)
        self.dense_layer = keras.layers.Dense(23, activation="softmax", dtype='float32')

    def call(self, inputs, training=False):
        x = self.base_model(inputs, training=training)
        x = self.global_max_pooling(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)

model = ConvNeXtLarge3Pretrained()

# Initialize with dummy
dummy = tf.zeros((1, 512, 512, 3))
_ = model(dummy)

# Load weights
model.load_weights("convnext_temp_model/model.weights.h5")


✅ Modelo carregado com sucesso!


# Make Predictions

In [ ]:
y_true, y_pred, y_prob = [], [], []

for images, labels in test_df:
    probs = model.predict(images, verbose=0)
    y_prob.extend(probs)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred) 
y_prob = np.array(y_prob)

accuracy = np.mean(y_true == y_pred)
print(f'Test accuracy: {accuracy:.4f}')

KeyboardInterrupt: 

# Classification Report

## 4. Classification report
> Primary metric for this project is **macro-F1** because classes may be imbalanced.
> Macro-F1 gives equal weight to all classes regardless of size.
> If macro-F1 is much lower than accuracy, the model is biased toward majority classes.

In [ ]:
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

# Save as file for the report
with open(OUTPUTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
print('Saved!')

## 5. Confusion matrix
> The diagonal = correct predictions. Off-diagonal = errors.
> Look for clusters of errors between visually similar classes.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    cmap='Blues',
    annot=True,
    fmt='d',
    linewidths=1,
    cbar=False,
    annot_kws={'fontsize': 10},
    yticklabels=class_names,
    xticklabels=class_names,
)
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# save figure for the report
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# Per-class accuracy
print('\nPer-class accuracy:')
for i, name in enumerate(class_names):
    per_class_acc = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0
    print(f'  {name:<30} {per_class_acc:.3f}')

## 7. Grad-CAM
> Visualises which regions of the painting the model focuses on.
> Reference: https://keras.io/examples/vision/grad_cam/
> For VGG16, the last conv layer is `block5_conv3`.

In [ ]:
inp = keras.Input(shape=(128, 128, 3))
x = model.rescaling(inp)
x = model.conv1(x)
x = model.conv2(x)
x = model.max_pooling1(x)
x = model.conv3(x)
x = model.conv4(x)
x = model.max_pooling2(x)
x = model.conv5(x)
x = model.conv6(x)
x = model.max_pooling3(x)
x = model.conv7(x)
conv8_out = model.conv8(x)
x = model.max_pooling4(conv8_out)
x = model.global_average_pooling(x)
x = model.dropout(x)
out = model.classifier(x)

grad_model = keras.Model(inputs=inp, outputs=[conv8_out, out])

In [ ]:
for images, labels in test_df.take(1):
    img_batch = tf.expand_dims(images[0], axis=0)
    img_np = (images[0].numpy()).astype('uint8')

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch, training=False)
        tape.watch(conv_outputs)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    heatmap_res = cv2.resize(heatmap, (128, 128))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_np, 0.6, heatmap_color, 0.4, 0)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.title(f"True: {class_names[np.argmax(labels[0])]}")
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title(f"Predicted: {class_names[pred_index]}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 8. Misclassified samples
> Show the worst predictions — highest confidence but wrong.
> Ask: are these images ambiguous? Similar to another class? Possibly mislabelled?

In [ ]:
# Collect all misclassified images with their confidence
wrong_imgs, wrong_true, wrong_pred, wrong_conf = [], [], [], []

for images, labels in test_df:
    probs        = model.predict(images, verbose=0)
    pred_classes = np.argmax(probs, axis=1)
    true_classes = np.argmax(labels.numpy(), axis=1)

    for i in range(len(true_classes)):
        if pred_classes[i] != true_classes[i]:
            wrong_imgs.append(images[i].numpy().astype('uint8'))
            wrong_true.append(true_classes[i])
            wrong_pred.append(pred_classes[i])
            wrong_conf.append(probs[i][pred_classes[i]])   # confidence of wrong prediction

# Sort by highest confidence wrong prediction (worst mistakes)
sorted_idx  = np.argsort(wrong_conf)[::-1]
n_show      = min(10, len(sorted_idx))

print(f'Total misclassified: {len(wrong_imgs)} / {len(y_true)} ({len(wrong_imgs)/len(y_true)*100:.1f}%)')

cols = 5
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
axes = axes.flatten()

for i, idx in enumerate(sorted_idx[:n_show]):
    axes[i].imshow(wrong_imgs[idx])
    axes[i].set_title(
        f'True: {class_names[wrong_true[idx]]}\nPred: {class_names[wrong_pred[idx]]}\nConf: {wrong_conf[idx]:.2f}',
        fontsize=8, color='red'
    )
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Most Confident Wrong Predictions', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'misclassified.png', dpi=150)
plt.show()

In [ ]:
print(f"{'Class Name':<30} | {'Total Errors'}")
print("-" * 45)

for i, name in enumerate(class_names):
    # Total images in this class minus the ones on the diagonal (correct)
    total_in_class = cm[i].sum()
    correct_preds = cm[i, i]
    errors = total_in_class - correct_preds
    
    print(f"{name:<30} | {int(errors)}")

In [ ]:
# Calculate error counts
error_counts = []
for i in range(len(class_names)):
    error_counts.append(cm[i].sum() - cm[i, i]) #cm is the confusion matrix

# Sort by error count
error_val_df = pd.DataFrame({'Artist': class_names, 'Errors': error_counts})
error_val_df = error_val_df.sort_values('Errors', ascending=False)

# Create Plot
plt.figure(figsize=(12, 8))
sns.barplot(x='Errors', y='Artist', data=error_val_df, palette='Reds_r') 

plt.title('Total Misclassifications per Artist (Class)', fontsize=14)
plt.xlabel('Number of Errors')
plt.ylabel('Artist (Class)')
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Save for your project folder
plt.savefig(FIGURES_DIR / 'errors_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# percentage of errors for each class

per_class_error_pct = []

for i in range(len(class_names)):
    total_samples = cm[i].sum()
    if total_samples > 0:
        # (Total images - correct predictions / total images) * 100
        error_pct = (1 - (cm[i,i]) / total_samples) * 100
        per_class_error_pct.append(error_pct)
    else:
        per_class_error_pct.append(0)

# Save as DataFrame to ease the visualization

error_df = pd.DataFrame({
    'Artists (Class)': class_names,
    'Error %': per_class_error_pct
}).sort_values('Error %', ascending=False) # Order by the classes with the bigger % of error

# Visualization
plt.figure(figsize=(12, 8))
ax = sns.barplot(data=error_df, x='Error %', y='Artists (Class)', palette='magma')

# To appear the % on the bars. 
for p in ax.patches:
    ax.annotate(f'{p.get_width():.1f}%', 
                (p.get_width(), p.get_y() + p.get_height() / 2),
                xytext=(5, 0), textcoords='offset points', 
                va='center', fontsize=9)

plt.title('Error Rate for each Artist (Class) (%)', fontsize=14)
plt.xlabel('Error Rate (%)')
plt.ylabel('Artist (Class)')
plt.xlim(0, 110) # text
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.savefig(FIGURES_DIR / 'error_percentage_by_class.png', dpi=150, bbox_inches='tight')
plt.show()